In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv("mental_health_risk_dataset (1).csv")

# Load into SQLite database
conn = sqlite3.connect('mental_health.db')
df.to_sql('survey', conn, if_exists='replace', index=False)
print("Data loaded into SQLite database successfully")

Data loaded into SQLite database successfully


In [2]:
query1 = """
SELECT 
    mental_health_risk,
    COUNT(*) as total_people,
    ROUND(AVG(sleep_hours), 2) as avg_sleep,
    ROUND(AVG(work_stress_level), 2) as avg_work_stress,
    ROUND(AVG(screen_time_hours_per_day), 2) as avg_screen_time
FROM survey
GROUP BY mental_health_risk
ORDER BY mental_health_risk
"""
result1 = pd.read_sql_query(query1, conn)
print("Average lifestyle metrics by risk level:")
print(result1)

Average lifestyle metrics by risk level:
   mental_health_risk  total_people  avg_sleep  avg_work_stress  \
0                   0          9357       7.16             5.03   
1                   1         11823       6.39             5.66   
2                   2          3820       5.31             6.29   

   avg_screen_time  
0             6.43  
1             6.53  
2             6.38  


In [3]:
query2 = """
SELECT 
    employment_status,
    mental_health_risk,
    COUNT(*) as count
FROM survey
GROUP BY employment_status, mental_health_risk
ORDER BY employment_status, mental_health_risk
"""
result2 = pd.read_sql_query(query2, conn)
print("Risk distribution by employment status:")
print(result2)

Risk distribution by employment status:
   employment_status  mental_health_risk  count
0           Employed                   0   2364
1           Employed                   1   2985
2           Employed                   2    944
3      Self-Employed                   0   2337
4      Self-Employed                   1   2948
5      Self-Employed                   2    960
6            Student                   0   2324
7            Student                   1   2962
8            Student                   2    941
9         Unemployed                   0   2332
10        Unemployed                   1   2928
11        Unemployed                   2    975


In [4]:
query3 = """
SELECT *
FROM survey
WHERE sleep_hours < 5 AND work_stress_level > 7
ORDER BY depression_score DESC
LIMIT 10
"""
result3 = pd.read_sql_query(query3, conn)
print("High-risk profiles: low sleep + high work stress:")
print(result3)

High-risk profiles: low sleep + high work stress:
   age  gender marital_status education_level employment_status  sleep_hours  \
0   28  Female         Single        Bachelor     Self-Employed          3.1   
1   35  Female       Divorced             PhD     Self-Employed          3.5   
2   40  Female        Married        Bachelor        Unemployed          4.9   
3   27   Other       Divorced          Master     Self-Employed          4.3   
4   32   Other       Divorced     High School          Employed          3.9   
5   56   Other        Married             PhD     Self-Employed          4.8   
6   56   Other         Single             PhD        Unemployed          3.9   
7   30   Other        Married             PhD          Employed          3.2   
8   47   Other         Single        Bachelor          Employed          3.5   
9   50   Other        Married     High School     Self-Employed          4.2   

   physical_activity_hours_per_week  screen_time_hours_per_day  \
0  

## INSIGHTS:
1. Average lifestyle metrics by risk level (Query 1):
This query evaluates how core daily habits scale alongside the model's target mental health risk classifications across all 25,000 records.
Risk Level          Total People          Avg. Sleep Hours          Avg. Work Stress          Avg. Screen Time
0 (Low Risk)         9,357                   7.16 hours               5.03                      10,6.43 hours
1 (Medium Risk)      11,823                  6.39 hours               5.66                      10,6.53 hours
2 (High Risk)        3,820                   5.31 hours               6.29                      10,6.38 hours

-A sharp, steady decline in sleep is visible as risk levels escalate. High-risk individuals (Class 2) average nearly 2 hours less sleep per night (5.31 hrs) than low-risk individuals (7.16 hrs), confirming a massive physiological dependency.
-Work stress scales upwards sequentially, jumping from an average of 5.03 to 6.29. This validates that cumulative occupational pressure directly corresponds to elevated risk classifications.
-Average screen time remains almost completely stagnant (~6.4 hours) across all three groups. This tells us that overall screen duration on its own is a weak differentiator, meaning the model likely prioritizes sleep and stress over flat screen hours.


2. Risk distribution by employment status (Query 2):
This matrix tracks how risk categories are distributed across different professional structures.
Employment status    Low risk (0)          Medium Risk (1)         HIght risk (2)            Total segment      High-Risk ratio
Employed              2,364                  2,985                     944                       6,293                15.00%
Self-Employed         2,337                  2,948                     960                       6,245                15.37%
Student               2,324                  2,962                     941                       6,227                15.11%
Unemployed            2,332                  2,928                     975                       6,235                15.63%
-The dataset shows a highly uniform distribution across all sectors. Every occupational category hovers almost identically around ~37% Low Risk, ~47% Medium Risk, and ~15% High Risk.
-While the distribution is tight, Unemployed individuals maintain the highest gross count (975) and marginal percentage (15.63%) of High Risk profiles, indicating that lack of structural routine or financial strain adds a subtle magnifying layer to mental health risks.

3.High-risk triage Profiling (Query 3):
This query performs direct anomaly detection, isolating the top 10 most acute individual profiles experiencing extreme compound strain (under 5 hours of sleep AND over 7/10 work stress) sorted by maximum clinical depression severity.

-Triage Severity Match: Out of the 10 filtered extreme cases, 80% (8 out of 10) are classified explicitly as Class 2 (High Risk). The remaining 2 profiles are safely flagged as Class 1 (Medium Risk). This provides empirical proof that the intersection of low sleep and extreme work stress acts as a highly reliable clinical indicator for acute system triage.

-Clinical Saturation: Every single individual in this top-10 anomaly pool hit a maximum clinical depression_score of 10, despite spanning completely different ages (27 to 56), genders, and educational levels. This confirms that when these behavioral boundaries break, clinical severity peaks regardless of demographic background.

